In [21]:
import tensorflow as tf
import numpy as np
import math

### Propagate uncertainty with Autograd

In [56]:
import jax.numpy as jnp
from jax import grad

def f(x, y):
    return x**2 * y + y**3

grad_f_x = grad(f, argnums=0)
grad_f_y = grad(f, argnums=1)

print(grad_f_x(2.0, 3.0))
print(grad_f_y(2.0, 3.0))

AttributeError: partially initialized module 'jax' has no attribute 'version' (most likely due to a circular import)

### Propagate uncertainty with tensorflow

**Compute derivative**

1. define func f  
2. set as tf.Variable values measured
3. use tf.gradienttape on them to compute derivative
4. set tool uncertainty  
5. use formula for final uncertainty
6. use f to compute central value

In [22]:
def f(x, y):
    return x**2 + y**2

x = tf.Variable(3.0) 
y = tf.Variable(4.0)

with tf.GradientTape(persistent=True) as tape:
    tape.watch([x, y])  # Ensure x and y are being watched
    z = f(x, y)

df_dx = tape.gradient(z, x)
df_dy = tape.gradient(z, y)

# print("Partial derivative with respect to x (df/dx):", df_dx.numpy())

In [33]:
# Sigma is tool uncertainty???

sigma_x = 0.1
sigma_y = 0.5

sigma_f = math.sqrt((df_dx * sigma_x)**2 + (df_dy * sigma_y)**2)  

print(f'f = {z} + {sigma_f:.3f}')

f = 25.0 + 4.045


### Do partial derivatives with sympy

In [59]:
from sympy import symbols, diff

x, y = symbols('x y')
def f(x, y):
    return x**2 + y**2

partial_x = diff(f, x)  # derivative with respect to x
partial_y = diff(f, y)  # derivative with respect to y

print(partial_x)  # Output: 2*x*y
print(partial_y)  # Output: x**2 + 3*y**2

value_df_dx = df_dx.subs({x: 2, y: 3}).evalf()

/usr/lib/python3/dist-packages/sympy/core/function.py:1233: SymPyDeprecationWarning: 

The string fallback in sympify() is deprecated.

To explicitly convert the string form of an object, use
sympify(str(obj)). To add define sympify behavior on custom
objects, use sympy.core.sympify.converter or define obj._sympy_
(see the sympify() docstring).

sympify() performed the string fallback resulting in the following string:

'<function f at 0x795e3c9db420>'

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-sympify-string-fallback
for details.

This has been deprecated since SymPy version 1.6. It
will be removed in a future version of SymPy.

  expr = sympify(expr)


SympifyError: Sympify of expression 'could not parse '<function f at 0x795e3c9db420>'' failed, because of exception being raised:
SyntaxError: invalid syntax (<string>, line 1)

### Uncertainty propagation, different?

In [24]:
# Define the model (change this for different fits)
def fit_model(x, a, b):
    return a * tf.exp(b * x)  # Example: Exponential function


# Define a function to compute the derivative of the model
# derivative is computed in a point x_val, with current parameters params

def compute_derivative(x_val, params):
    with tf.GradientTape() as tape:
        tape.watch(x_val)
        y_pred = fit_model(x_val, *params)
    return tape.gradient(y_pred, x_val)  # Compute dy/dx

In [25]:
# How it works?

# params = [m_hat, q_hat] 
# x_val = tf.Variable(0.5)  # ad esempio un punto specifico
# dy_dx = compute_derivative(x_val, params)

**Uncertainty propagation not-independent measures**

In [57]:
import numpy as np
from sympy import symbols, diff

def propagate_uncertainty(f, variables, uncertainties, covariance_matrix, values):
    """
    Computes the uncertainty propagation for dependent variables.
    
    Parameters:
        f: sympy expression
            The function of variables for which uncertainty propagation is calculated.
        variables: list of sympy symbols
            The variables in the function.
        uncertainties: list of floats
            The standard deviations (uncertainties) of the variables.
        covariance_matrix: 2D numpy array
            The covariance matrix for the variables.
        values: list of floats
            The numerical values of the variables to evaluate the derivatives.
            
    Returns:
        float: The propagated uncertainty of the function.
    """
    # Compute partial derivatives
    partial_derivatives = [diff(f, var) for var in variables]
    
    # Substitute values into the partial derivatives
    partial_derivatives_numeric = [
        pd.subs(dict(zip(variables, values))).evalf() for pd in partial_derivatives
    ]
    
    # Evaluate the uncertainty propagation formula
    uncertainty_squared = 0
    for i in range(len(variables)):
        for j in range(len(variables)):
            uncertainty_squared += (
                partial_derivatives_numeric[i] *
                partial_derivatives_numeric[j] *
                covariance_matrix[i, j]
            )
    
    return np.sqrt(float(uncertainty_squared))

# Example usage

# Define symbols
x, y = symbols('x y')

# Define the function
f = x**2 + y**2

# Define variables, uncertainties, covariance matrix, and values
variables = [x, y]
uncertainties = [0.1, 0.2]
covariance_matrix = np.array([[0.1**2, 0.02], [0.02, 0.2**2]])
values = [2.0, 3.0]  # Values of x and y

# Compute propagated uncertainty
result = propagate_uncertainty(f, variables, uncertainties, covariance_matrix, values)
print(f"Propagated uncertainty: {result}")


Propagated uncertainty: 1.6


In [61]:
import numpy as np
from sympy import symbols, diff

def propagate_error(f, variables, uncertainties, covariance_matrix, values):
    """
    Computes the uncertainty propagation for independent variables.
    
    Parameters:
        f: sympy expression
            The function of variables for which uncertainty propagation is calculated.
        variables: list of sympy symbols
            The variables in the function.
        uncertainties: list of floats
            The standard deviations (uncertainties) of the variables.
        covariance_matrix: 2D numpy array
            The covariance matrix for the variables.
        values: list of floats
            The numerical values of the variables to evaluate the derivatives.
            
    Returns:
        float: The propagated uncertainty of the function.
    """
    # Compute partial derivatives
    partial_derivatives = [diff(f, var) for var in variables]
    
    # Substitute values into the partial derivatives
    partial_derivatives_numeric = [
        pd.subs(dict(zip(variables, values))).evalf() for pd in partial_derivatives
    ]
    
    # Evaluate the uncertainty propagation formula
    uncertainty_squared = 0
    for i in range(len(variables)):
        for j in range(len(variables)):
            uncertainty_squared += (
                partial_derivatives_numeric[i] *
                partial_derivatives_numeric[j] *
                covariance_matrix[i, j]
            )
    
    return np.sqrt(float(uncertainty_squared))

# Example usage

# Define symbols
x, y = symbols('x y')

# Define the function
f = x**2 + y**2

# Define variables, uncertainties, covariance matrix, and values
variables = [x, y]
uncertainties = [0.1, 0.2]
covariance_matrix = np.array([[0.1**2, 0.02], [0.02, 0.2**2]])
values = [2.0, 3.0]  # Values of x and y

# Compute propagated uncertainty
result = propagate_uncertainty(f, variables, uncertainties, covariance_matrix, values)
print(f"Propagated uncertainty: {result}")


Propagated uncertainty: 1.6


In [70]:
import tensorflow as tf

def propagation(f, param_values, param_uncertainties):
    """
    f: function to evaluate, f(x, *params)
    x_val: value for x (can be None if your f has no x)
    param_values: list or tuple of parameter values
    param_uncertainties: list or tuple of parameter uncertainties
    """
    param_values_tf = [tf.constant(p, dtype=tf.float32) for p in param_values]
    param_uncertainties_tf = [tf.constant(u, dtype=tf.float32) for u in param_uncertainties]

    with tf.GradientTape(persistent=True) as tape:
        tape.watch(param_values_tf)
        y = f(*param_values_tf)

    # Compute all partial derivatives
    gradients = tape.gradient(y, param_values_tf)

    # Propagate uncertainties
    total_variance = tf.add_n([
        (grad**2) * (unc**2) for grad, unc in zip(gradients, param_uncertainties_tf)
    ])

    uncertainty_y = tf.sqrt(total_variance)

    return y.numpy(), uncertainty_y.numpy()


In [72]:
def my_model(a, b):
    return a * tf.exp(b)

param_values = [2.0, 0.5]  # a=2.0, b=0.5
param_uncertainties = [0.1, 0.02]  # Δa=0.1, Δb=0.02

value, uncertainty = propagation(my_model, param_values, param_uncertainties)

print("Function value =", value)
print("Uncertainty =", uncertainty)

Function value = 3.2974424
Uncertainty = 0.17757273
